# R-Bot Industrial — LoRA de intents (Colab GPU)

Entrena **Qwen2.5-1.5B-Instruct** con LoRA (FP16, sin bitsandbytes) para mapear órdenes en español → JSON de intents.

> Nota: evitamos QLoRA/4-bit porque en Colab reciente `bitsandbytes` choca con CUDA/triton. El modelo 1.5B cabe en T4 en FP16.

## Antes de empezar
1. **Runtime → Change runtime type → T4 GPU**
2. Ejecuta las celdas **en orden**
3. Si re-subes este notebook, usa **Runtime → Disconnect and delete runtime** y vuelve a conectar GPU

## Después (PC)
```bash
ollama create rbot-intent -f ml/export/Modelfile.rbot-intent
# .env: OLLAMA_MODEL=rbot-intent
bash ~/Documents/Proyectos/start-rbot.sh
```


In [ ]:
# 0) GPU
!nvidia-smi
import torch
assert torch.cuda.is_available(), "Runtime → Change runtime type → GPU (T4)"
print("OK:", torch.cuda.get_device_name(0), "| VRAM", round(torch.cuda.get_device_properties(0).total_memory/1e9,1), "GB")


In [ ]:
# 1) Dependencias (sin bitsandbytes — más estable en Colab 2025/2026)
%pip install -q -U "transformers>=4.44,<4.48" "peft>=0.12,<0.14" \
  "datasets>=2.20,<3" "trl>=0.9,<0.12" "accelerate>=0.33,<1.1" \
  sentencepiece einops
# HF_TOKEN es opcional (solo warning). Qwen2.5-1.5B es público.


In [ ]:
# 2) Dataset
from pathlib import Path
import json

# True = subir intents_es.jsonl | False = dataset embebido (111 ejemplos)
UPLOAD = False

INTENT_ACTIONS = {
  "list_topics": "ros2 topic list",
  "list_nodes": "ros2 node list",
  "list_services": "ros2 service list",
  "list_actions": "ros2 action list",
  "navigate": "send_navigation_goal",
  "cancel_navigation": "cancel_navigation",
  "return_home": "return_home",
  "unknown": "unknown_action",
}

def to_sft(row):
    intent = row.get("intent") or "unknown"
    dest = row.get("destination")
    params = {}
    if intent == "navigate" and dest:
        params["destination"] = dest
    out = {
        "intent": intent,
        "action": INTENT_ACTIONS.get(intent, "unknown_action"),
        "parameters": params,
        "confidence": 0.95 if intent != "unknown" else 0.3,
    }
    return {"instruction": row["text"], "output": json.dumps(out, ensure_ascii=False)}

# JSON válido → dicts Python (null → None)
EMBEDDED = json.loads(r"""[{"text": "muéstrame los topics", "intent": "list_topics", "destination": null}, {"text": "lista los topics", "intent": "list_topics", "destination": null}, {"text": "qué topics hay", "intent": "list_topics", "destination": null}, {"text": "dame los topics ros", "intent": "list_topics", "destination": null}, {"text": "topics activos", "intent": "list_topics", "destination": null}, {"text": "enséñame los topics del robot", "intent": "list_topics", "destination": null}, {"text": "show topics", "intent": "list_topics", "destination": null}, {"text": "list ros topics", "intent": "list_topics", "destination": null}, {"text": "quiero ver los topics", "intent": "list_topics", "destination": null}, {"text": "cuáles son los topics", "intent": "list_topics", "destination": null}, {"text": "muéstrame los nodos", "intent": "list_nodes", "destination": null}, {"text": "qué nodos hay activos", "intent": "list_nodes", "destination": null}, {"text": "lista nodos", "intent": "list_nodes", "destination": null}, {"text": "nodos ros2", "intent": "list_nodes", "destination": null}, {"text": "show nodes", "intent": "list_nodes", "destination": null}, {"text": "list nodes", "intent": "list_nodes", "destination": null}, {"text": "dame los nodos del sistema", "intent": "list_nodes", "destination": null}, {"text": "nodos activos ahora", "intent": "list_nodes", "destination": null}, {"text": "lista servicios", "intent": "list_services", "destination": null}, {"text": "muéstrame los services", "intent": "list_services", "destination": null}, {"text": "qué servicios hay", "intent": "list_services", "destination": null}, {"text": "list services", "intent": "list_services", "destination": null}, {"text": "lista actions", "intent": "list_actions", "destination": null}, {"text": "muéstrame las actions", "intent": "list_actions", "destination": null}, {"text": "qué acciones ros hay", "intent": "list_actions", "destination": null}, {"text": "list actions", "intent": "list_actions", "destination": null}, {"text": "avanza", "intent": "navigate", "destination": "adelante"}, {"text": "adelante", "intent": "navigate", "destination": "adelante"}, {"text": "avanza un poco", "intent": "navigate", "destination": "adelante"}, {"text": "avanza en línea recta", "intent": "navigate", "destination": "adelante"}, {"text": "muévete hacia adelante", "intent": "navigate", "destination": "adelante"}, {"text": "go forward", "intent": "navigate", "destination": "adelante"}, {"text": "drive forward", "intent": "navigate", "destination": "adelante"}, {"text": "camina hacia adelante", "intent": "navigate", "destination": "adelante"}, {"text": "sigue recto", "intent": "navigate", "destination": "adelante"}, {"text": "avanza despacio", "intent": "navigate", "destination": "adelante"}, {"text": "ve a la estación a", "intent": "navigate", "destination": "estación A"}, {"text": "ve a la estación b", "intent": "navigate", "destination": "estación B"}, {"text": "ve a revisar la fuga de la bomba 3", "intent": "navigate", "destination": "fuga bomba 3"}, {"text": "inspecciona la bomba 3", "intent": "navigate", "destination": "bomba 3"}, {"text": "dirígete al tanque 2", "intent": "navigate", "destination": "tanque 2"}, {"text": "ve a la zona de peligros", "intent": "navigate", "destination": "zona peligros"}, {"text": "revisa la fuga", "intent": "navigate", "destination": "fuga"}, {"text": "ve a la entrada", "intent": "navigate", "destination": "entrada"}, {"text": "navega a home", "intent": "navigate", "destination": "home"}, {"text": "undock", "intent": "navigate", "destination": "undock"}, {"text": "desatraca", "intent": "navigate", "destination": "undock"}, {"text": "detén", "intent": "cancel_navigation", "destination": null}, {"text": "detén el robot", "intent": "cancel_navigation", "destination": null}, {"text": "para", "intent": "cancel_navigation", "destination": null}, {"text": "parar", "intent": "cancel_navigation", "destination": null}, {"text": "frena", "intent": "cancel_navigation", "destination": null}, {"text": "stop", "intent": "cancel_navigation", "destination": null}, {"text": "halt", "intent": "cancel_navigation", "destination": null}, {"text": "cancela la navegación", "intent": "cancel_navigation", "destination": null}, {"text": "deja de moverte", "intent": "cancel_navigation", "destination": null}, {"text": "quieto", "intent": "cancel_navigation", "destination": null}, {"text": "stop robot", "intent": "cancel_navigation", "destination": null}, {"text": "regresa a la base", "intent": "return_home", "destination": null}, {"text": "vuelve a casa", "intent": "return_home", "destination": null}, {"text": "ve a home", "intent": "return_home", "destination": null}, {"text": "dock", "intent": "return_home", "destination": null}, {"text": "atraca", "intent": "return_home", "destination": null}, {"text": "regresa al dock", "intent": "return_home", "destination": null}, {"text": "return home", "intent": "return_home", "destination": null}, {"text": "vuelve a la base", "intent": "return_home", "destination": null}, {"text": "carga la batería en base", "intent": "return_home", "destination": null}, {"text": "hola cómo estás", "intent": "unknown", "destination": null}, {"text": "qué hora es", "intent": "unknown", "destination": null}, {"text": "cuéntame un chiste", "intent": "unknown", "destination": null}, {"text": "receta de pasta", "intent": "unknown", "destination": null}, {"text": "who are you", "intent": "unknown", "destination": null}, {"text": "necesito que el robot avance un metro", "intent": "navigate", "destination": "adelante"}, {"text": "por favor detén la marcha", "intent": "cancel_navigation", "destination": null}, {"text": "quiero ver qué nodos están corriendo", "intent": "list_nodes", "destination": null}, {"text": "enséñame el listado de topics", "intent": "list_topics", "destination": null}, {"text": "el operario pide retorno a base", "intent": "return_home", "destination": null}, {"text": "ve a chequear la fuga de gas", "intent": "navigate", "destination": "fuga de gas"}, {"text": "enséñame qué topics están publicados", "intent": "list_topics", "destination": null}, {"text": "puedes listar los topics del create3", "intent": "list_topics", "destination": null}, {"text": "necesito el listado de topics ros2", "intent": "list_topics", "destination": null}, {"text": "dime los nodos que están vivos", "intent": "list_nodes", "destination": null}, {"text": "qué nodos ros hay corriendo", "intent": "list_nodes", "destination": null}, {"text": "muéstrame services disponibles", "intent": "list_services", "destination": null}, {"text": "qué actions tiene el robot", "intent": "list_actions", "destination": null}, {"text": "acércate a la bomba 3", "intent": "navigate", "destination": "bomba 3"}, {"text": "ve a inspeccionar el tanque 1", "intent": "navigate", "destination": "tanque 1"}, {"text": "dirígete a la estación de carga", "intent": "navigate", "destination": "estación de carga"}, {"text": "ve a la zona de válvulas", "intent": "navigate", "destination": "zona de válvulas"}, {"text": "revisa la fuga de gas en el área B", "intent": "navigate", "destination": "fuga de gas área B"}, {"text": "navega hasta el laboratorio", "intent": "navigate", "destination": "laboratorio"}, {"text": "ve al pasillo principal", "intent": "navigate", "destination": "pasillo principal"}, {"text": "muévete hacia la salida de emergencia", "intent": "navigate", "destination": "salida de emergencia"}, {"text": "ve a chequear el sensor de presión", "intent": "navigate", "destination": "sensor de presión"}, {"text": "avanza medio metro", "intent": "navigate", "destination": "adelante"}, {"text": "sigue hacia adelante", "intent": "navigate", "destination": "adelante"}, {"text": "forward please", "intent": "navigate", "destination": "adelante"}, {"text": "go to station a", "intent": "navigate", "destination": "estación A"}, {"text": "inspect pump 3 leak", "intent": "navigate", "destination": "fuga bomba 3"}, {"text": "para ya", "intent": "cancel_navigation", "destination": null}, {"text": "frena el create3", "intent": "cancel_navigation", "destination": null}, {"text": "cancelar misión", "intent": "cancel_navigation", "destination": null}, {"text": "emergency stop de navegación", "intent": "cancel_navigation", "destination": null}, {"text": "vuelve al dock", "intent": "return_home", "destination": null}, {"text": "atraca en la base", "intent": "return_home", "destination": null}, {"text": "regresa a la estación de carga", "intent": "return_home", "destination": null}, {"text": "go home", "intent": "return_home", "destination": null}, {"text": "cuál es la capital de francia", "intent": "unknown", "destination": null}, {"text": "pon música", "intent": "unknown", "destination": null}, {"text": "abre youtube", "intent": "unknown", "destination": null}, {"text": "traduce esto al inglés", "intent": "unknown", "destination": null}]""")

if UPLOAD:
    from google.colab import files
    print("Sube intents_es.jsonl …")
    uploaded = files.upload()
    name = next(iter(uploaded))
    rows = [json.loads(l) for l in Path(name).read_text(encoding="utf-8").splitlines() if l.strip()]
else:
    rows = EMBEDDED
    print("Usando dataset embebido")

sft = [to_sft(r) for r in rows]
Path("intents_sft.jsonl").write_text("\n".join(json.dumps(x, ensure_ascii=False) for x in sft), encoding="utf-8")
print(f"{len(sft)} ejemplos SFT")
print("Ejemplo:", sft[0])


In [ ]:
# 3) Modelo FP16 + LoRA (sin 4-bit / bitsandbytes)
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model

BASE = "Qwen/Qwen2.5-1.5B-Instruct"
OUT_DIR = "rbot-intent-lora"

assert torch.cuda.is_available(), "Activa GPU T4"
dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

tokenizer = AutoTokenizer.from_pretrained(BASE, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE,
    torch_dtype=dtype,
    device_map="auto",
    trust_remote_code=True,
)
model.config.use_cache = False
model.gradient_checkpointing_enable()

model = get_peft_model(model, LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
))
model.print_trainable_parameters()
print("dtype:", dtype, "| device:", next(model.parameters()).device)


In [ ]:
# 4) Dataset chat
from datasets import load_dataset

SYSTEM = '''Eres el intérprete de control del robot R-Bot (Create3 + ROS2).
Responde ÚNICAMENTE con JSON válido (sin markdown):
{"intent":"...","action":"...","parameters":{},"confidence":0.0}
Intents: list_topics, list_nodes, list_services, list_actions, navigate, cancel_navigation, return_home, unknown.
Si navigate: parameters.destination es obligatorio (ej: "adelante", "bomba 3").'''

def format_example(ex):
    messages = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": ex["instruction"]},
        {"role": "assistant", "content": ex["output"]},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return {"text": text}

ds = load_dataset("json", data_files="intents_sft.jsonl", split="train")
ds = ds.map(format_example).train_test_split(test_size=0.12, seed=42)
print(ds)
print(ds["train"][0]["text"][:350], "…")


In [ ]:
# 5) Entrenar (~10–25 min en T4)
from trl import SFTTrainer
from transformers import TrainingArguments

# Compat trl antiguo/nuevo
try:
    from trl import SFTConfig
    use_sft_config = True
except ImportError:
    use_sft_config = False

common = dict(
    output_dir=OUT_DIR,
    num_train_epochs=8,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    logging_steps=5,
    save_steps=100,
    fp16=(dtype == torch.float16),
    bf16=(dtype == torch.bfloat16),
    optim="adamw_torch",
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    report_to=[],
    max_grad_norm=0.3,
    gradient_checkpointing=True,
)

# eval_strategy vs evaluation_strategy según versión
try:
    args = TrainingArguments(**common, eval_strategy="epoch")
except TypeError:
    args = TrainingArguments(**common, evaluation_strategy="epoch")

if use_sft_config:
    # trl nuevo prefiere SFTConfig; mantenemos API clásica si falla
    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=ds["train"],
        eval_dataset=ds["test"],
        dataset_text_field="text",
        max_seq_length=512,
        args=args,
        packing=False,
    )
else:
    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=ds["train"],
        eval_dataset=ds["test"],
        dataset_text_field="text",
        max_seq_length=512,
        args=args,
        packing=False,
    )

trainer.train()
trainer.model.save_pretrained(OUT_DIR)
tokenizer.save_pretrained(OUT_DIR)
print("Adaptador →", OUT_DIR)


In [ ]:
# 6) Smoke test
def ask(text, max_new=128):
    messages = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": text},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new, do_sample=False)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True).strip()

for q in [
    "muéstrame los topics",
    "avanza un poco",
    "ve a revisar la fuga de la bomba 3",
    "detén el robot",
    "regresa a la base",
    "cuéntame un chiste",
]:
    print("Q:", q)
    print("A:", ask(q))
    print("---")


In [ ]:
# 7) Merge LoRA → pesos únicos (para export / Ollama)
from peft import PeftModel

# El modelo actual ya es PeftModel en GPU; fusionar y guardar
merged = model.merge_and_unload()
MERGE_DIR = "rbot-intent-merged"
merged.save_pretrained(MERGE_DIR, safe_serialization=True)
tokenizer.save_pretrained(MERGE_DIR)
print("Merge →", MERGE_DIR)


In [ ]:
# 8) Guardar en Google Drive
from google.colab import drive
import shutil
from pathlib import Path

drive.mount("/content/drive")
dest = Path("/content/drive/MyDrive/rbot-industrial-ml")
dest.mkdir(parents=True, exist_ok=True)
for name in ["rbot-intent-lora", "rbot-intent-merged", "intents_sft.jsonl"]:
    src = Path(name)
    if not src.exists():
        continue
    target = dest / src.name
    if target.exists():
        shutil.rmtree(target) if target.is_dir() else target.unlink()
    if src.is_dir():
        shutil.copytree(src, target)
    else:
        shutil.copy2(src, target)
    print("→", target)
print("Listo en Drive/rbot-industrial-ml/")


## 9) (Opcional) Export GGUF en Colab

Solo si tienes RAM/disco. Si falla, en el PC puedes seguir usando el Modelfile con `FROM qwen2.5:1.5b` + SYSTEM, o convertir el merge localmente.

```python
# %cd /content
# !git clone --depth 1 https://github.com/ggerganov/llama.cpp
# %cd llama.cpp
# !pip install -q -r requirements.txt
# !python convert_hf_to_gguf.py /content/rbot-intent-merged --outfile /content/rbot-intent-f16.gguf
# !cmake -B build && cmake --build build --target llama-quantize -j
# !./build/bin/llama-quantize /content/rbot-intent-f16.gguf /content/rbot-intent-q4_k_m.gguf Q4_K_M
```

Copia el `.gguf` a `ml/export/`, edita `Modelfile.rbot-intent` (`FROM ./rbot-intent-q4_k_m.gguf`) y:

```bash
ollama create rbot-intent -f ml/export/Modelfile.rbot-intent
```


In [ ]:
# 10) Qué hacer en el PC
print('''
1. En Drive descarga rbot-intent-merged/ (o el GGUF).
2. Local ya tienes rbot-intent (SYSTEM + qwen). Tras GGUF:
     ollama create rbot-intent -f ml/export/Modelfile.rbot-intent
3. apps/api/.env → OLLAMA_MODEL=rbot-intent
4. bash ~/Documents/Proyectos/start-rbot.sh
5. Prueba: "ve a la fuga de la bomba 3" / "muéstrame los topics"
''')
